## MLflow 3.16 Best Practices and MLOps Workflow

This notebook walks through a complete MLOps cycle:

- **Experiment Tracking:** parameters, metrics and artifacts (confusion matrix, prediction
  samples) logged for every run, so any result can be reproduced or audited.
- **Dataset Versioning:** the exact data behind each run, recorded with the MLflow Dataset API.
- **Model Registry:** registering the Classifier and promoting a version with the
  `@champion` alias.
- **Serving:** an API that loads whichever version currently holds `@champion`.

Everything runs in containers. Nothing is installed on your machine.

# Class II: Infrastructure as Code for MLOps

## 🌱 Project: Bonsai Species Classifier for Plant E-commerce

Welcome to our hands-on MLOps session! We're building a **bonsai species classifier** for a plant website that will:
- **Identify bonsai species** from plant measurements
- **Provide care recommendations** based on species type
- **Help customers** choose the right bonsai for their needs

### Infrastructure Stack (All Containerized!)
- **MLflow**: For experiment tracking and model registry
- **Docker**: All services are containerized (no local installation needed!)
- **JupyterLab**: You're running this from a container right now
- **API**: Model serving for the plant website

## Quick Setup Check
Make sure all containers are running:
- MLflow UI: http://localhost:5001 (track bonsai model experiments)
- JupyterLab: http://localhost:8888 (you're here!)
- API: http://localhost:8080 (bonsai species prediction service)

Let's start building our bonsai classifier! 🌳

In [1]:
# The environment is already built. Nothing to install here.
#
# This container was built from docker/Dockerfile.jupyter with every version pinned in
# docker/requirements-notebook.txt. That is what Infrastructure as Code means in practice:
# the environment is a file in the repository, not a command someone remembered to run.
import mlflow, sklearn, pandas as pd, numpy as np

print(f"mlflow       {mlflow.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"pandas       {pd.__version__}")
print(f"numpy        {np.__version__}")

# MLFLOW_TRACKING_URI comes from docker-compose.yml. "mlflow" is the service name, which
# Compose resolves as a hostname on the shared network.
print(f"\ntracking URI: {mlflow.get_tracking_uri()}")

mlflow       3.16.1
scikit-learn 1.9.1
pandas       3.0.6
numpy        2.5.3

tracking URI: http://mlflow:5000


# 🌳 Bonsai Species Classification with MLflow

Now let's train our bonsai species classifier and track the experiment. We'll classify 4 types of bonsai:
- **Juniper Bonsai** (0): Hardy, needle-like foliage
- **Ficus Bonsai** (1): Broad leaves, aerial roots  
- **Pine Bonsai** (2): Long needles, rugged bark
- **Maple Bonsai** (3): Distinctive lobed leaves

We'll track:
- **Parameters**: Model configuration (n_estimators, etc.)
- **Metrics**: Classification performance (accuracy, etc.)
- **Artifacts**: The trained bonsai classifier model

Check the MLflow UI at http://localhost:5001 to see your bonsai classification experiments! 🌱

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import numpy as np

# NOTE: this is synthetic data, not real bonsai measurements. We generate it so the class
# has something separable to train on; any accuracy we report says something about the
# generator, not about botany. Swapping in a real dataset is left as homework.
X, y = make_classification(
    n_samples=300,
    n_features=4,
    n_classes=4,
    n_informative=4,
    n_redundant=0,
    random_state=42,
)

feature_names = ['leaf_length_cm', 'leaf_width_cm', 'branch_thickness_mm', 'height_cm']
bonsai_species = ['Juniper', 'Ficus', 'Pine', 'Maple']

# Shift and scale each column so the numbers at least look like plant measurements.
# make_classification returns roughly standard-normal values, so these are approximately
# mean +/- 3*scale, not hard bounds.
X[:, 0] = X[:, 0] * 0.5 + 2.0    # leaf length, cm       (~2.0 +/- 1.5)
X[:, 1] = X[:, 1] * 0.3 + 1.5    # leaf width, cm        (~1.5 +/- 0.9)
X[:, 2] = X[:, 2] * 2.0 + 5.0    # branch thickness, mm  (~5.0 +/- 6.0)
X[:, 3] = X[:, 3] * 10.0 + 25.0  # height, cm            (~25 +/- 30)

bonsai_df = pd.DataFrame(X, columns=feature_names)
bonsai_df['species'] = [bonsai_species[i] for i in y]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Bonsai dataset created:")
print(f"  training samples: {X_train.shape[0]}")
print(f"  test samples:     {X_test.shape[0]}")
print(f"  features:         {feature_names}")
print(f"  species:          {bonsai_species}")
print("\nSample measurements:")
print(bonsai_df.head())

Bonsai dataset created:
  training samples: 240
  test samples:     60
  features:         ['leaf_length_cm', 'leaf_width_cm', 'branch_thickness_mm', 'height_cm']
  species:          ['Juniper', 'Ficus', 'Pine', 'Maple']

Sample measurements:
   leaf_length_cm  leaf_width_cm  branch_thickness_mm  height_cm species
0        1.380904       1.188396             1.650313  19.065886   Ficus
1        1.893503       1.425476             5.341917  30.000305   Maple
2        2.007036       1.334003             7.963199  24.248633   Maple
3        2.892563       1.966694             5.784386  14.109763   Maple
4        3.342822       2.035872             5.564911   9.854393   Maple


# 🆕 MLflow 3.16 Dataset Feature
MLflow Datasets allow you to track, version, and reuse input data for your experiments. This ensures reproducibility and makes it easy to compare results across different runs.
- **Track the exact data used for each run**
- **Version datasets for auditability**
- **Share and reuse datasets in future experiments**
Let's log our bonsai dataset using MLflow's new Dataset API.

In [3]:
# Experiments with different configurations to compare performance
import mlflow
import mlflow.sklearn
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import mlflow.data
from datetime import datetime

date = datetime.today().strftime('%Y-%m-%d')

# First thing version our dataset
# Written to /tmp, not into the repository: generated data is an artifact,
# and MLflow is where it gets versioned.
DATASET_PATH = f"/tmp/bonsai_dataset_{date}.csv"
bonsai_df.to_csv(DATASET_PATH, index=False)

dataset = mlflow.data.from_pandas(
    bonsai_df,
    source=DATASET_PATH,
    name=f"bonsai_species_measurements_{date}",
    targets="species" if "species" in bonsai_df.columns else None,
)
# Set up experiment for bonsai classification
mlflow.set_experiment("Bonsai-Species-Classification")

# List of configurations to test
experiment_configs = [
    {
        "name": "baseline_model",
        "n_estimators": 50,
        "max_depth": 3,
        "min_samples_split": 2,
        "description": "Baseline model with conservative settings"
    },
    {
        "name": "balanced_model", 
        "n_estimators": 100,
        "max_depth": 5,
        "min_samples_split": 3,
        "description": "Balanced model between complexity and performance"
    },
    {
        "name": "complex_model",
        "n_estimators": 200,
        "max_depth": 8,
        "min_samples_split": 2,
        "description": "More complex model for maximum performance"
    },
    {
        "name": "optimized_model",
        "n_estimators": 150,
        "max_depth": 6,
        "min_samples_split": 4,
        "description": "Optimized model based on previous results"
    }
]

print("🧪 Running multiple experiments for comparison...")
print("=" * 60)

# Run experiments
experiment_results = []

for config in experiment_configs:
    with mlflow.start_run(run_name=config["name"]):

        # Log the bonsai dataset using MLflow Datasets
        mlflow.log_input(dataset, context="training")

        
        # Train model with specific configuration
        bonsai_classifier = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            min_samples_split=config["min_samples_split"],
            random_state=42
        )
        

        bonsai_classifier.fit(X_train, y_train)
        preds = bonsai_classifier.predict(X_test)
        print("✅ Bonsai dataset logged as MLflow Dataset!")
        
        # Calculate detailed metrics
        accuracy = accuracy_score(y_test, preds)
        precision = precision_score(y_test, preds, average='weighted')
        recall = recall_score(y_test, preds, average='weighted')
        f1 = f1_score(y_test, preds, average='weighted')
        
        # Log parameters
        mlflow.log_params({
            "n_estimators": config["n_estimators"],
            "max_depth": config["max_depth"],
            "min_samples_split": config["min_samples_split"],
            "model_type": "RandomForestClassifier",
            "dataset": "Bonsai Species",
            "n_species": len(bonsai_species),
            "description": config["description"]
        })
        
        # Log metrics
        mlflow.log_metrics({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "test_samples": len(y_test),
            "training_samples": len(y_train)
        })
        
        # Log model
        # MLflow now serializes scikit-learn models with `skops` instead of pickle.
        # skops refuses to save types it cannot verify, and sklearn.tree._tree.Tree — the
        # node storage behind every RandomForest — is one of them: it holds raw node
        # indices that .predict() follows without bounds checking, so a tampered file can
        # crash or read out of bounds. We vouch for this type explicitly.
        model_info = mlflow.sklearn.log_model(
            bonsai_classifier,
            name="bonsai_classifier",
            signature=mlflow.models.infer_signature(X_train, y_train),
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        # In MLflow 3 a logged model is its own entity with its own id, no longer just a
        # folder of artifacts hanging off the run. Recording the id here lets us register
        # exactly this model later, instead of asking the run to guess which one we meant.
        mlflow.set_tag("logged_model_id", model_info.model_id)

        
        # Log confusion matrix as artifact
        cm = confusion_matrix(y_test, preds)
        fig, ax = plt.subplots(figsize=(6, 4))
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title('Confusion Matrix')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.colorbar(im)
        plt.xticks(np.arange(len(bonsai_species)), bonsai_species)
        plt.yticks(np.arange(len(bonsai_species)), bonsai_species)
        for i in range(len(bonsai_species)):
            for j in range(len(bonsai_species)):
                ax.text(j, i, cm[i, j], ha='center', va='center', color='black')
        plt.tight_layout()
        cm_path = f"/tmp/confusion_matrix_{config['name']}.png"
        plt.savefig(cm_path)
        plt.close(fig)
        mlflow.log_artifact(cm_path)
        os.remove(cm_path)

        # Log sample predictions as CSV artifact
        sample_df = pd.DataFrame({
            'actual': [bonsai_species[i] for i in y_test],
            'predicted': [bonsai_species[i] for i in preds]
        })
        sample_path = f"/tmp/sample_predictions_{config['name']}.csv"
        sample_df.to_csv(sample_path, index=False)
        mlflow.log_artifact(sample_path)
        os.remove(sample_path)

        # Save results for comparison
        experiment_results.append({
            "name": config["name"],
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "description": config["description"]
        })
        
        print(f"✅ {config['name']}: Accuracy={accuracy:.3f}, F1={f1:.3f}")

print("\n📊 Experiment Summary:")
print("-" * 60)
for result in sorted(experiment_results, key=lambda x: x['accuracy'], reverse=True):
    print(f"🏆 {result['name']}: {result['accuracy']:.3f} acc | {result['f1_score']:.3f} f1")
    print(f"   📝 {result['description']}")

print(f"\n🌐 Compare experiments in MLflow UI: http://localhost:5001")
print("💡 Use 'Compare' to view differences side by side!")

2026/09/22 18:53:06 INFO mlflow.tracking.fluent: Experiment with name 'Bonsai-Species-Classification' does not exist. Creating a new experiment.


🧪 Running multiple experiments for comparison...
✅ Bonsai dataset logged as MLflow Dataset!
✅ baseline_model: Accuracy=0.617, F1=0.621
🏃 View run baseline_model at: http://mlflow:5000/#/experiments/1/runs/a15e42648af34e99928fb3cf456a5ca5
🧪 View experiment at: http://mlflow:5000/#/experiments/1
✅ Bonsai dataset logged as MLflow Dataset!
✅ balanced_model: Accuracy=0.750, F1=0.753
🏃 View run balanced_model at: http://mlflow:5000/#/experiments/1/runs/add6767d36c543ffa09aadb3025459e0
🧪 View experiment at: http://mlflow:5000/#/experiments/1
✅ Bonsai dataset logged as MLflow Dataset!
✅ complex_model: Accuracy=0.767, F1=0.762
🏃 View run complex_model at: http://mlflow:5000/#/experiments/1/runs/dbef46c93dd44e4796959db399a5403d
🧪 View experiment at: http://mlflow:5000/#/experiments/1
✅ Bonsai dataset logged as MLflow Dataset!
✅ optimized_model: Accuracy=0.817, F1=0.817
🏃 View run optimized_model at: http://mlflow:5000/#/experiments/1/runs/58b29cd348e4404585f49a5ba71f584b
🧪 View experiment at: ht

# MLflow Model Registry

The **Model Registry** is the single place that answers "which version is live?".

### What it gives you
- **Versioning**: every registered model gets an incrementing version number
- **Aliases**: a moveable pointer such as `@champion` or `@candidate` aimed at one version
- **Tags and descriptions**: metadata for humans
- **Lineage**: each version links back to the run that produced it

### Aliases, not stages
Older MLflow used fixed *stages* (`Staging`, `Production`). They are deprecated. An alias
does the same job without the fixed vocabulary: you decide what `@champion` means, you can
have as many aliases as you need, and moving one is a single call.

The API in this stack loads `models:/Bonsai-Species-Classifier@champion`. Move the alias,
and what the API serves changes — without touching the API.

### The workflow
1. **Experiment** — several runs, compare them
2. **Register** — the best run's model becomes version N
3. **Validate** — point `@candidate` at it and test it
4. **Promote** — move `@champion` to it
5. **Monitor** — watch it, and repeat

In [4]:
# Register the best model and mark it as a candidate for promotion.
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = mlflow.get_experiment_by_name("Bonsai-Species-Classification")
best_run = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1,
)[0]

print("Best run:")
print(f"  run id:   {best_run.info.run_id}")
print(f"  accuracy: {best_run.data.metrics['accuracy']:.3f}")
print(f"  f1:       {best_run.data.metrics['f1_score']:.3f}")
print(f"  params:   {best_run.data.params}")

# The name identifies the model. It does NOT encode where the model is in its lifecycle —
# that is what the alias is for. A model called "...-Production" becomes a lie the moment
# you register a version that is not in production.
model_name = "Bonsai-Species-Classifier"
# Register the exact logged model, by id. Using "runs:/<run>/bonsai_classifier" still
# works, but MLflow has to resolve it and warns that the run "has no artifacts at" that
# path -- an alarming message for what is really just the MLflow 2 spelling.
model_uri = f"models:/{best_run.data.tags['logged_model_id']}"

model_version = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f"\nRegistered {model_name} version {model_version.version}")

client.set_model_version_tag(model_name, model_version.version, "use_case", "plant_ecommerce_classification")
client.set_model_version_tag(model_name, model_version.version, "model_type", "RandomForestClassifier")

client.update_model_version(
    name=model_name,
    version=model_version.version,
    description=f"""Bonsai Species Classifier for e-commerce.

Performance:
  accuracy  {best_run.data.metrics['accuracy']:.3f}
  precision {best_run.data.metrics['precision']:.3f}
  recall    {best_run.data.metrics['recall']:.3f}
  f1        {best_run.data.metrics['f1_score']:.3f}

Configuration:
  n_estimators      {best_run.data.params['n_estimators']}
  max_depth         {best_run.data.params['max_depth']}
  min_samples_split {best_run.data.params['min_samples_split']}

Classes: Juniper (0), Ficus (1), Pine (2), Maple (3)
Data: synthetic, generated by sklearn.make_classification""",
)

# @candidate means "this one is next in line, go and test it".
client.set_registered_model_alias(model_name, "candidate", model_version.version)
print(f"Alias @candidate -> version {model_version.version}")

print(f"\nInspect it: http://localhost:5001/#/models/{model_name}")

Successfully registered model 'Bonsai-Species-Classifier'.
2026/09/22 19:08:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Bonsai-Species-Classifier, version 1


Best run:
  run id:   58b29cd348e4404585f49a5ba71f584b
  accuracy: 0.817
  f1:       0.817
  params:   {'n_estimators': '150', 'max_depth': '6', 'min_samples_split': '4', 'model_type': 'RandomForestClassifier', 'dataset': 'Bonsai Species', 'n_species': '4', 'description': 'Optimized model based on previous results'}


Created version '1' of model 'Bonsai-Species-Classifier'.



Registered Bonsai-Species-Classifier version 1
Alias @candidate -> version 1

Inspect it: http://localhost:5001/#/models/Bonsai-Species-Classifier


### A note on stages, which you will meet everywhere else

Nearly every MLflow tutorial, and most of the code you will inherit, promotes models like
this instead:

```python
client.transition_model_version_stage(name=model_name, version=1, stage="Production")
model = mlflow.pyfunc.load_model("models:/Bonsai-Species-Classifier/Production")
```

Those calls still run in MLflow 3.16, but they emit a `FutureWarning`: stages were
deprecated in 2.9. They imposed four fixed names on everyone, and only one version could
hold each. Aliases replace them with pointers you name yourself.

Recognise the old form when you see it. Do not write new code with it.

## Validate the candidate

Before a version becomes the one customers hit, load it back out of the registry exactly
as the API will, and check it behaves. Loading it *from the registry* rather than reusing
the object still in memory is the point: it proves the artifact round-tripped correctly.

In [5]:
# Load the candidate the same way the API will load the champion.
model_name = "Bonsai-Species-Classifier"
candidate_uri = f"models:/{model_name}@candidate"

print(f"Loading {candidate_uri}")
candidate_model = mlflow.pyfunc.load_model(candidate_uri)

predictions = candidate_model.predict(X_test)
candidate_accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy of the loaded candidate on the test set: {candidate_accuracy:.3f}")
print(f"First test sample predicted as: {bonsai_species[int(predictions[0])]}")

# A promotion gate: refuse to promote anything that does not clear the bar.
THRESHOLD = 0.70
promote = candidate_accuracy >= THRESHOLD
print(f"\nGate: accuracy >= {THRESHOLD} -> {'PASS' if promote else 'FAIL'}")

Loading models:/Bonsai-Species-Classifier@candidate
Accuracy of the loaded candidate on the test set: 0.817
First test sample predicted as: Maple

Gate: accuracy >= 0.7 -> PASS


## Promote the candidate to champion

Moving the `@champion` alias is the deployment. The API is not rebuilt, redeployed or
restarted — it asks the registry for `@champion` and gets the new version.

This is also how a rollback works: point `@champion` back at the previous version.

In [6]:
# Promote: move @champion onto the candidate version.
client = MlflowClient()
model_name = "Bonsai-Species-Classifier"

candidate = client.get_model_version_by_alias(model_name, "candidate")

if not promote:
    print(f"Not promoting: version {candidate.version} did not pass the gate.")
else:
    try:
        previous = client.get_model_version_by_alias(model_name, "champion")
        print(f"Current champion: version {previous.version}")
    except Exception:
        print("No champion yet — this will be the first.")

    client.set_registered_model_alias(model_name, "champion", candidate.version)
    print(f"Alias @champion -> version {candidate.version}")
    print("\nThe API now serves this version. Nothing was redeployed.")

No champion yet — this will be the first.
Alias @champion -> version 1

The API now serves this version. Nothing was redeployed.


# What we did

- Built and compared four Classifier configurations, tracked in MLflow
- Versioned the dataset behind every run
- Logged metrics and artifacts so any run can be reproduced
- Registered the best model and validated it through the registry
- Promoted it by moving the `@champion` alias — no redeploy

## Next
- Serve it: the cell below calls the API, which loads `@champion`
- Automate it: the next class puts this cycle behind CI/CD
- Monitor it: watch for drift, retrain, move the alias again

In [7]:
# Call the API. It loads models:/Bonsai-Species-Classifier@champion on first use.
import requests

features = [2.1, 1.8, 5.5, 25.3]  # leaf_length_cm, leaf_width_cm, branch_thickness_mm, height_cm

# "api" is the service name from docker-compose.yml, resolved on the shared network.
# From your own browser the same service is at http://localhost:8080
try:
    response = requests.post("http://api:8080/predict", json={"features": features}, timeout=60)
    response.raise_for_status()
    print(response.json())
except requests.HTTPError as e:
    print(f"HTTP {e.response.status_code}: {e.response.text}")
    print("\nIf this is a 503, no version holds the @champion alias yet — run the cell above.")
except Exception as e:
    print(f"API call failed: {e}")

# If you promote a NEW version later, tell the API to pick it up:
#   requests.post("http://api:8080/reload")

{'prediction': 3, 'species': 'Maple', 'care_recommendations': 'Needs partial shade, consistent moisture, protection from wind', 'input_features': {'leaf_length_cm': 2.1, 'leaf_width_cm': 1.8, 'branch_thickness_mm': 5.5, 'height_cm': 25.3}}
